In [ ]:
"""
Step 01: Download IBKR 1-Minute Bars Into The Staging Folder

Objective:

    - Download SPY 1-minute TRADES bars (regular trading hours) from IBKR, one request per NYSE session, several sessions
      in flight, every session checked minute by minute against the NYSE schedule.
    - Write one file per session into the STAGING folder (store01_rawzone/ibkr_spy_1min_staging/), never into the raw
      folder that the research pipeline reads. Every attempt is appended to <staging>/download_log.csv.

Prerequisites:

    - IB Gateway (or TWS) running and logged in, API enabled on config.IBKR_PORT_INT (4001 = Gateway live).
    - The network share of nicodesktop reachable (config.DATA_ROOT_PATH_STR).

Outputs:

    - <staging>/ohlcv_data_YYYYMMDD.csv (one per downloaded session), <staging>/download_log.csv.

Next: notebooks/step03_raw_data_check.ipynb (or scripts/merge_staging_into_raw.py) to check the staged sessions and add
them to the raw folder. The same download runs from the command line: python scripts/download_ibkr_ohlcv.py
"""
None

In [ ]:
# IMPORT THE INGEST CONFIGURATION
from ingest import config
# READ WHETHER THE SPY RAW FOLDER NAME AND so.paths DISAGREE
folder_name_problem_str = config.get_folder_name_problem_str()
# STOP WHEN THEY DISAGREE (NOTHING IS WRITTEN AND IBKR IS NOT CONNECTED)
if folder_name_problem_str:
    # PRINT THE MESSAGE
    print(folder_name_problem_str)
    # STOP
    raise SystemExit(folder_name_problem_str)
# IMPORT GENERAL PACKAGES
import pandas as pd
# IMPORT THE IBKR CLIENT
from ingest.ibkr_client import IbkrApp
# IMPORT THE DOWNLOAD FUNCTIONS
from ingest.ibkr_download import get_download_session_pdf, download_session_pdf
# IMPORT THE MARKET STATUS FUNCTION
from ingest.ibkr_stream import get_market_status_dict
# IMPORT THE MAINTENANCE FUNCTIONS
from ingest.raw_maintenance import get_raw_issue_pdf, merge_staging_into_raw_pdf

In [ ]:
# DEFINE THE SESSIONS TO DOWNLOAD (None, None = EVERY CLOSED SESSION AFTER THE LAST ONE IN RAW + STAGING)
date1_str, date2_str = None, None
# DEFINE EXPLICIT DATES INSTEAD OF A RANGE (E.G. ["2009-07-27", "2013-12-23"]; [] = USE THE RANGE)
date_str_list = []
# DEFINE WHETHER SESSIONS ALREADY PRESENT ARE DOWNLOADED AGAIN (INTO STAGING ONLY; RAW IS NEVER REPLACED)
redownload_bool = False
# DISPLAY THE FOLDERS
print(f"Raw folder:\t{config.RAW_OHLCV_PATH_STR}")
print(f"Staging folder:\t{config.STAGING_OHLCV_PATH_STR}")

In [ ]:
# COLLECT THE MARKET STATUS (REAL NEW YORK TIME)
market_status_dict = get_market_status_dict()
# DISPLAY INFORMATION
pd.Series(market_status_dict)

In [ ]:
# CONNECT (REUSE THE CONNECTION IF THIS CELL WAS ALREADY RUN)
try:
    # IF THE APPLICATION EXISTS AND IS CONNECTED
    connected_bool = app.isConnected() or app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_NOTEBOOK_CLIENT_ID_INT)
# IF THE APPLICATION DOES NOT EXIST YET
except NameError:
    # CREATE AND CONNECT IT
    app = IbkrApp()
    connected_bool = app.connect_app(config.IBKR_HOST_STR, config.IBKR_PORT_INT, config.IBKR_NOTEBOOK_CLIENT_ID_INT)

In [ ]:
# SELECT THE SESSIONS TO DOWNLOAD
session_pdf = get_download_session_pdf(date1_str, date2_str, date_str_list, redownload_bool)
# DISPLAY INFORMATION
print(f"Sessions to download: {len(session_pdf):,}")
# PREVIEW DATAFRAME
session_pdf

In [ ]:
# DOWNLOAD THE SESSIONS INTO THE STAGING FOLDER
download_summary_pdf = download_session_pdf(app, session_pdf)
# PREVIEW DATAFRAME
download_summary_pdf

In [ ]:
# READ THE DOWNLOAD LOG
download_log_pdf = pd.read_csv(config.DOWNLOAD_LOG_FILE_PATH_STR) if not session_pdf.empty else pd.DataFrame()
# PREVIEW THE LAST ATTEMPTS
download_log_pdf.tail(20)

In [ ]:
# CHECK THE STAGED SESSIONS TOGETHER WITH THE RAW FOLDER (READ ONLY)
issue_pdf = get_raw_issue_pdf([config.RAW_OHLCV_PATH_STR, config.STAGING_OHLCV_PATH_STR], session_pdf["date"].min() if not session_pdf.empty else None)
# PREVIEW DATAFRAME
issue_pdf

In [ ]:
# DRY RUN OF THE MERGE INTO THE RAW FOLDER (NOTHING IS WRITTEN; USE step03 OR THE SCRIPT WITH --apply TO ADD THE FILES)
merge_check_pdf = merge_staging_into_raw_pdf(apply_bool_in=False)
# PREVIEW DATAFRAME
merge_check_pdf

In [ ]:
# DISCONNECT
app.disconnect_app()